In [95]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, datasets, models
import torch.nn.functional as F
from PIL import Image
import onnxruntime as ort
import numpy as np
import torch.onnx

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#device = torch.device("cpu")
print("Using device:", device)

# 1. Hyperparameters
num_classes = 3 # 3 for 1000 da and 2000 da bill and unknown the 500 da is not included
num_epochs = 10
batch_size = 32
learning_rate = 0.0001

# 2. Transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),                # resize to 224x224
    transforms.Grayscale(num_output_channels=3),  # convert gray -> 3-channel
    transforms.RandomHorizontalFlip(p=0.5),        # data augmentation 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])
])


Using device: cuda


In [ ]:

# Create datasets
root_dir = "dataset/" # Update your Path pls
train_dataset = datasets.ImageFolder(root=os.path.join(root_dir, 'training'),
                                     transform=train_transform)
val_dataset   = datasets.ImageFolder(root=os.path.join(root_dir, 'validation'),
                                     transform=val_transform)

In [ ]:
# Data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print("Number of training samples:", len(train_dataset))
print("Number of validation samples:", len(val_dataset))
print("Class names:", train_dataset.classes)

# Load pretrained mobilenet_v2 : mobilenet_v2 is a well known cnn model used for edge AI
model = models.mobilenet_v2(pretrained=True)

Number of training samples: 5091
Number of validation samples: 1111
Class names: ['1000', '2000', 'unknown']


In [68]:
in_features = model.classifier[1].in_features
in_features

1280

In [69]:
model.classifier[1] = nn.Sequential(
    nn.Linear(in_features, num_classes),
    nn.Softmax(dim=1)  # Apply Softmax across classes
)
model = model.to(device)

In [ ]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
# Training loop
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
   
    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = 100.0 * correct / total

    # Validation
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for val_images, val_labels in val_loader:
            val_images, val_labels = val_images.to(device), val_labels.to(device)
            val_outputs = model(val_images)
            v_loss = criterion(val_outputs, val_labels)
           
            val_loss += v_loss.item() * val_images.size(0)
            _, val_predicted = torch.max(val_outputs, 1)
            val_total += val_labels.size(0)
            val_correct += (val_predicted == val_labels).sum().item()
   
    val_epoch_loss = val_loss / len(val_dataset)
    val_epoch_acc = 100.0 * val_correct / val_total

    print(f"Epoch [{epoch+1}/{num_epochs}], "
          f"Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.2f}%, "
          f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.2f}%")

print("Training complete.")

Epoch [1/10], Train Loss: 0.6018, Train Acc: 96.62%, Val Loss: 0.5539, Val Acc: 99.91%
Epoch [2/10], Train Loss: 0.5570, Train Acc: 99.65%, Val Loss: 0.5532, Val Acc: 100.00%
Epoch [3/10], Train Loss: 0.5537, Train Acc: 99.84%, Val Loss: 0.5536, Val Acc: 99.82%
Epoch [4/10], Train Loss: 0.5565, Train Acc: 99.69%, Val Loss: 0.5522, Val Acc: 100.00%
Epoch [5/10], Train Loss: 0.5546, Train Acc: 99.80%, Val Loss: 0.5522, Val Acc: 100.00%
Epoch [6/10], Train Loss: 0.5533, Train Acc: 99.86%, Val Loss: 0.5523, Val Acc: 99.91%
Epoch [7/10], Train Loss: 0.5521, Train Acc: 99.96%, Val Loss: 0.5525, Val Acc: 99.91%
Epoch [8/10], Train Loss: 0.5525, Train Acc: 99.90%, Val Loss: 0.5523, Val Acc: 99.91%
Epoch [9/10], Train Loss: 0.5572, Train Acc: 99.55%, Val Loss: 0.5519, Val Acc: 100.00%
Epoch [10/10], Train Loss: 0.5530, Train Acc: 99.90%, Val Loss: 0.5520, Val Acc: 100.00%
Training complete.


In [ ]:

torch.save(model.state_dict(), "Mobile_net_V2.pth")
print("Model saved to Mobile_net_V2.pth")

Model saved to Mobile_net_V2.pth


In [ ]:

# Manual test of your model
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])
])

# Load image
img_path = "test.jpg"
image = Image.open(img_path)

# Apply transforms
image = test_transform(image)

# Add a batch dimension: (C, H, W) -> (1, C, H, W)
image = image.unsqueeze(0)

# Move to device if using GPU
image = image.to(device)

# Make prediction
model.eval()
with torch.no_grad():
    outputs = model(image)
    _, predicted_class = torch.max(outputs, 1)
    class_index = predicted_class.item()
print(outputs)

class_names = ["1000", "2000", "unknown"]
predicted_label = class_names[class_index]
print("Predicted class:", predicted_label)
# Move to CPU, flatten, convert to list and round
clean_output = [round(x.item(), 4) for x in outputs.squeeze().cpu()]
# Print
print(*clean_output)

tensor([[0.0041, 0.0771, 0.9188]], device='cuda:0')
Predicted class: unknown
0.0041 0.0771 0.9188


In [47]:
image.shape

torch.Size([1, 3, 224, 224])

In [ ]:
# Save model in onnx format you can further use it in C++ for higher performance
# Define model and load trained weights
resnet_bill = model  # Replace with your model class
resnet_bill.load_state_dict(torch.load("Mobile_net_V2.pth", map_location=torch.device('cpu')))
resnet_bill.eval()  # Set to evaluation mode

onnx_path = "Mobile_net_V2.onnx"
# Export the model to ONNX
torch.onnx.export(
    resnet_bill, 
    image, 
    onnx_path,  # Save as ONNX file
    export_params=True,  # Save trained weights
    opset_version=11,  # Ensure compatibility
    do_constant_folding=True,  # Optimize constant expressions
    input_names=["input"], 
    output_names=["output"]
)

print("Model saved as model.onnx")


Model saved as model.onnx


In [ ]:
# test your ONNX trained model
import torchvision.transforms as transforms
from PIL import Image

# Load ONNX model
onnx_path = "rpi_V2.onnx"
session = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])  # Use "CUDAExecutionProvider" for GPU

# Define preprocessing (should match training)
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Load and preprocess the image
img_path = "test.jpg"
image = Image.open(img_path)
image = image_transform(image).unsqueeze(0).numpy()  # Convert to NumPy (ONNX requires NumPy arrays)

# Run inference
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name
output = session.run([output_name], {input_name: image})[0]  # Output is already a NumPy array
# Class labels
class_labels = ["1000", "2000","unknown"]

# Ensure `output` is a NumPy array before calling argmax
predicted_class = np.argmax(output, axis=1)[0]  # Get class index
predicted_label = class_labels[predicted_class]

print(f"Predicted Class: {predicted_label}")